# ToM Parameter Identification - Enhanced Contrast Pair Methodology

This notebook implements an **improved methodology** using contrast pair datasets to identify ToM-sensitive parameters.

## Key Improvements Over Original Paper

1. **Contrast Pairs Instead of ToM vs C4**:
   - Compare high-ToM vs low-ToM questions on the same scenarios
   - Better content control (same scenario, different reasoning requirements)
   - More balanced token supervision

2. **Two Datasets**:
   - `simpletom_contrast_pairs.json`: False belief (high ToM) vs factual (low ToM)
   - `self_other.json`: Self-reference vs other-reference (ToM aspect)

## Epistemic Status on Dataset Quality

**High confidence** that these datasets are methodologically superior:
- ✓ Better content control (minimal confounds)
- ✓ Balanced token counts between conditions
- ✓ Isolates specific cognitive mechanism (mental state reasoning)

**Moderate confidence** on practical effectiveness:
- ✓ Should identify more precise ToM-specific parameters
- ⚠ May identify fewer parameters (more selective)
- ⚠ Dampening effectiveness depends on parameter overlap with production ToM circuits
- ⚠ Need empirical validation to confirm dampening works in practice

**Key Question**: Will perturbing these parameters actually dampen ToM in interactive use?
- Best case: Clean isolation of ToM-specific circuits → reliable dampening
- Worst case: ToM is too distributed → perturbation affects general language ability
- Most likely: Partial dampening with some collateral effects on related reasoning

In [ ]:
# Setup and imports
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import subprocess
from typing import List, Dict, Tuple

# Configuration - MODIFY THESE AS NEEDED
BASE_MODEL = "meta-llama/Llama-3.2-1B"  # Small LLaMA variant for testing
CACHE_DIR = "./cache"
PERMANENT_STORAGE = "./permanent_storage"
TMP_DIR = "./tmp"

# Gradient computation settings
NSAMPLES = 500  # More samples for better FIM estimate (can use more since these are shorter)
SEQLEN = 0      # Use full sequence length

# Evaluation settings
M_VALUES = [0.0, 1e-5, 2e-5, 3e-5, 4e-5, 5e-5, 1e-4]  # Test slightly higher m too
EVAL_REPS = 5
BATCH_SIZE = 64
MAX_MODEL_LEN = 1024
TENSOR_PARALLEL_SIZE = 1

# Dataset paths
SIMPLETOM_CONTRAST = "./tom_dataset/simpletom_contrast_pairs.json"
SELF_OTHER_CONTRAST = "./self_other_dataset/self_other.json"
TOM_TASKS_FILE = "./ToM_tasks.py"

# Create directories
os.makedirs(PERMANENT_STORAGE, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"✓ Configuration loaded")
print(f"  Model: {BASE_MODEL}")
print(f"  Samples per condition: {NSAMPLES}")
print(f"  M values: {M_VALUES}")

## Dataset Exploration

**Epistemic Status**: High confidence - just loading and inspecting data.

In [ ]:
# Load and explore SimpleTOM contrast pairs
with open(SIMPLETOM_CONTRAST, 'r') as f:
    simpletom_data = json.load(f)

print(f"SimpleTOM Contrast Pairs Dataset")
print(f"  Total pairs: {len(simpletom_data)}")
print(f"\nExample pair:")
example = simpletom_data[0]
print(f"  ID: {example['id']}")
print(f"  Category: {example['category']}")
print(f"  Scenario: {example['scenario'][:100]}...")
print(f"\n  High ToM prompt: {example['high_tom_prompt'][:150]}...")
print(f"  High ToM answer: {example['high_tom_completion']}")
print(f"\n  Low ToM prompt: {example['low_tom_prompt'][:150]}...")
print(f"  Low ToM answer: {example['low_tom_completion']}")
print(f"\n  Requires false belief: {example['metadata']['requires_false_belief']}")

# Calculate token length distribution
high_tom_lengths = [len(item['high_tom_combined']) for item in simpletom_data[:100]]
low_tom_lengths = [len(item['low_tom_combined']) for item in simpletom_data[:100]]

print(f"\nCharacter length statistics (first 100 samples):")
print(f"  High ToM: mean={np.mean(high_tom_lengths):.1f}, std={np.std(high_tom_lengths):.1f}")
print(f"  Low ToM:  mean={np.mean(low_tom_lengths):.1f}, std={np.std(low_tom_lengths):.1f}")
print(f"  Difference: {abs(np.mean(high_tom_lengths) - np.mean(low_tom_lengths)):.1f} chars")

In [ ]:
# Load and explore self/other dataset
with open(SELF_OTHER_CONTRAST, 'r') as f:
    self_other_data = json.load(f)

print(f"Self/Other Contrast Pairs Dataset")
print(f"  Total pairs: {len(self_other_data)}")
print(f"\nExample pair:")
example = self_other_data[0]
print(f"  Self-subject:  {example['self_subject'][:200]}...")
print(f"  Other-subject: {example['other_subject'][:200]}...")

# Calculate length distribution
self_lengths = [len(item['self_subject']) for item in self_other_data[:100]]
other_lengths = [len(item['other_subject']) for item in self_other_data[:100]]

print(f"\nCharacter length statistics (first 100 samples):")
print(f"  Self:  mean={np.mean(self_lengths):.1f}, std={np.std(self_lengths):.1f}")
print(f"  Other: mean={np.mean(other_lengths):.1f}, std={np.std(other_lengths):.1f}")
print(f"  Difference: {abs(np.mean(self_lengths) - np.mean(other_lengths)):.1f} chars")

## Prepare Datasets for Gradient Computation

**Epistemic Status**: High confidence. Simple data transformation to format expected by `create_gradient.py`.

We need to create datasets with `{"txt": "..."}` format. For each contrast pair, we'll create two separate datasets:
- High condition (high ToM or self-reference)
- Low condition (low ToM or other-reference)

In [ ]:
# Prepare SimpleTOM datasets
def prepare_simpletom_datasets():
    """Convert SimpleTOM contrast pairs into separate high/low datasets."""
    
    with open(SIMPLETOM_CONTRAST, 'r') as f:
        data = json.load(f)
    
    # Shuffle and take subset
    np.random.seed(42)
    indices = np.random.permutation(len(data))[:NSAMPLES]
    
    high_tom = [{"txt": data[i]["high_tom_combined"]} for i in indices]
    low_tom = [{"txt": data[i]["low_tom_combined"]} for i in indices]
    
    # Save to temp files
    high_tom_path = os.path.join(TMP_DIR, "simpletom_high_tom.json")
    low_tom_path = os.path.join(TMP_DIR, "simpletom_low_tom.json")
    
    with open(high_tom_path, 'w') as f:
        json.dump(high_tom, f, indent=2)
    
    with open(low_tom_path, 'w') as f:
        json.dump(low_tom, f, indent=2)
    
    print(f"✓ SimpleTOM datasets prepared")
    print(f"  High ToM: {len(high_tom)} samples -> {high_tom_path}")
    print(f"  Low ToM:  {len(low_tom)} samples -> {low_tom_path}")
    
    return high_tom_path, low_tom_path

simpletom_high_path, simpletom_low_path = prepare_simpletom_datasets()

In [ ]:
# Prepare self/other datasets
def prepare_self_other_datasets():
    """Convert self/other contrast pairs into separate datasets."""
    
    with open(SELF_OTHER_CONTRAST, 'r') as f:
        data = json.load(f)
    
    # Shuffle and take subset
    np.random.seed(42)
    indices = np.random.permutation(len(data))[:NSAMPLES]
    
    self_ref = [{"txt": data[i]["self_subject"]} for i in indices]
    other_ref = [{"txt": data[i]["other_subject"]} for i in indices]
    
    # Save to temp files
    self_path = os.path.join(TMP_DIR, "self_reference.json")
    other_path = os.path.join(TMP_DIR, "other_reference.json")
    
    with open(self_path, 'w') as f:
        json.dump(self_ref, f, indent=2)
    
    with open(other_path, 'w') as f:
        json.dump(other_ref, f, indent=2)
    
    print(f"✓ Self/Other datasets prepared")
    print(f"  Self:  {len(self_ref)} samples -> {self_path}")
    print(f"  Other: {len(other_ref)} samples -> {other_path}")
    
    return self_path, other_path

self_path, other_path = prepare_self_other_datasets()

## Compute Gradients for Contrast Pairs

**Epistemic Status**: High confidence in approach, but requires GPU.

### SimpleTOM Contrast: High ToM vs Low ToM

This will identify parameters sensitive to mental state reasoning vs factual reasoning.

In [ ]:
# Define gradient checkpoint paths for SimpleTOM
SIMPLETOM_HIGH_GRAD = os.path.join(PERMANENT_STORAGE, "gradients", "simpletom_high_tom")
SIMPLETOM_LOW_GRAD = os.path.join(PERMANENT_STORAGE, "gradients", "simpletom_low_tom")

os.makedirs(os.path.dirname(SIMPLETOM_HIGH_GRAD), exist_ok=True)
os.makedirs(os.path.dirname(SIMPLETOM_LOW_GRAD), exist_ok=True)

print(f"SimpleTOM gradient checkpoints:")
print(f"  High ToM: {SIMPLETOM_HIGH_GRAD}")
print(f"  Low ToM:  {SIMPLETOM_LOW_GRAD}")

In [ ]:
# Compute gradients for SimpleTOM contrast
def compute_contrast_gradients(data_path, output_path, name):
    """Compute squared gradients for a dataset."""
    if os.path.exists(os.path.join(output_path, "config.json")):
        print(f"✓ {name} gradients already exist at {output_path}")
        return
    
    cmd = [
        "python", "create_gradient.py",
        "--model", BASE_MODEL,
        "--dataset", "tom",  # Use tom mode (supervises last token)
        "--data_path", data_path,
        "--nsamples", str(NSAMPLES),
        "--seqlen", str(SEQLEN),
        "--cache_dir", CACHE_DIR,
        "--out", output_path
    ]
    
    print(f"Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"ERROR: {result.stderr}")
        raise RuntimeError(f"{name} gradient computation failed")
    
    print(f"✓ {name} gradients saved to {output_path}")
    print(result.stdout)

# Uncomment to run (requires GPU)
# compute_contrast_gradients(simpletom_high_path, SIMPLETOM_HIGH_GRAD, "SimpleTOM High ToM")
# compute_contrast_gradients(simpletom_low_path, SIMPLETOM_LOW_GRAD, "SimpleTOM Low ToM")

### Self/Other Contrast

**Epistemic Status**: Moderate confidence on practical utility.

This identifies parameters sensitive to self-reference vs other-reference. This may capture:
- Perspective-taking mechanisms
- First-person vs third-person reasoning
- Self-awareness aspects of ToM

In [ ]:
# Define gradient checkpoint paths for self/other
SELF_GRAD = os.path.join(PERMANENT_STORAGE, "gradients", "self_reference")
OTHER_GRAD = os.path.join(PERMANENT_STORAGE, "gradients", "other_reference")

os.makedirs(os.path.dirname(SELF_GRAD), exist_ok=True)
os.makedirs(os.path.dirname(OTHER_GRAD), exist_ok=True)

print(f"Self/Other gradient checkpoints:")
print(f"  Self:  {SELF_GRAD}")
print(f"  Other: {OTHER_GRAD}")

In [ ]:
# Compute gradients for self/other contrast
# Uncomment to run (requires GPU)
# compute_contrast_gradients(self_path, SELF_GRAD, "Self-reference")
# compute_contrast_gradients(other_path, OTHER_GRAD, "Other-reference")

## Chunk Gradients

**Epistemic Status**: High confidence.

In [ ]:
# Define chunk directories
SIMPLETOM_HIGH_CHUNKS = os.path.join(TMP_DIR, "chunks", "simpletom_high")
SIMPLETOM_LOW_CHUNKS = os.path.join(TMP_DIR, "chunks", "simpletom_low")
SELF_CHUNKS = os.path.join(TMP_DIR, "chunks", "self")
OTHER_CHUNKS = os.path.join(TMP_DIR, "chunks", "other")

for d in [SIMPLETOM_HIGH_CHUNKS, SIMPLETOM_LOW_CHUNKS, SELF_CHUNKS, OTHER_CHUNKS]:
    os.makedirs(d, exist_ok=True)

In [ ]:
# Chunk all gradients
def chunk_gradients(grad_checkpoint, output_dir, name):
    """Split gradient checkpoint into per-layer chunks."""
    if os.path.exists(os.path.join(output_dir, "manifest.json")):
        print(f"✓ {name} chunks already exist")
        return
    
    cmd = [
        "python", "chunk_gradient.py",
        "--model", grad_checkpoint,
        "--output_path", output_dir,
        "--cache_dir", CACHE_DIR,
        "--device_map", "cpu"
    ]
    
    print(f"Chunking {name}...")
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"ERROR: {result.stderr}")
        raise RuntimeError(f"{name} chunking failed")
    
    print(f"✓ {name} chunks saved")

# Uncomment to run
# chunk_gradients(SIMPLETOM_HIGH_GRAD, SIMPLETOM_HIGH_CHUNKS, "SimpleTOM High")
# chunk_gradients(SIMPLETOM_LOW_GRAD, SIMPLETOM_LOW_CHUNKS, "SimpleTOM Low")
# chunk_gradients(SELF_GRAD, SELF_CHUNKS, "Self")
# chunk_gradients(OTHER_GRAD, OTHER_CHUNKS, "Other")

## Run Evaluations

**Epistemic Status**: High confidence in implementation.

We'll run two separate evaluations:
1. **SimpleTOM contrast**: Mask = (high_tom_grad in top-m) AND (low_tom_grad NOT in top-m)
2. **Self/Other contrast**: Mask = (self_grad in top-m) AND (other_grad NOT in top-m)

In [ ]:
# Define evaluation output directories
EVAL_SIMPLETOM = os.path.join(PERMANENT_STORAGE, "evaluation_results", "simpletom_contrast")
EVAL_SELF_OTHER = os.path.join(PERMANENT_STORAGE, "evaluation_results", "self_other_contrast")

os.makedirs(EVAL_SIMPLETOM, exist_ok=True)
os.makedirs(EVAL_SELF_OTHER, exist_ok=True)

print(f"Evaluation output directories:")
print(f"  SimpleTOM: {EVAL_SIMPLETOM}")
print(f"  Self/Other: {EVAL_SELF_OTHER}")

In [ ]:
# Run SimpleTOM contrast evaluation
def run_evaluation(high_chunks, low_chunks, output_dir, experiment_name):
    """Run ToM and perplexity evaluation."""
    m_list = ",".join(str(m) for m in M_VALUES)
    
    cmd = [
        "python", "ToM_and_perplexity_evaluation.py",
        "--model", BASE_MODEL,
        "--grad_tom_chunks", high_chunks,  # "ToM-sensitive" condition
        "--grad_c4_chunks", low_chunks,    # "Control" condition
        "--tom_tasks", TOM_TASKS_FILE,
        "--out_dir", output_dir,
        "--cache_dir", CACHE_DIR,
        "--tensor_parallel_size", str(TENSOR_PARALLEL_SIZE),
        "--max_model_len", str(MAX_MODEL_LEN),
        "--batch_size", str(BATCH_SIZE),
        "--reps", str(EVAL_REPS),
        "--m_list", m_list
    ]
    
    print(f"\n{'='*80}")
    print(f"Running {experiment_name} evaluation")
    print(f"{'='*80}")
    print(f"Command: {' '.join(cmd)}")
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"ERROR: {result.stderr}")
        raise RuntimeError(f"{experiment_name} evaluation failed")
    
    print(f"✓ {experiment_name} evaluation complete")
    print(result.stdout)

# Uncomment to run (requires GPU + vLLM)
# run_evaluation(SIMPLETOM_HIGH_CHUNKS, SIMPLETOM_LOW_CHUNKS, EVAL_SIMPLETOM, "SimpleTOM Contrast")
# run_evaluation(SELF_CHUNKS, OTHER_CHUNKS, EVAL_SELF_OTHER, "Self/Other Contrast")

## Analysis & Comparison

**Epistemic Status**: High confidence in analysis code, moderate confidence in predictions.

### Expected Results

**SimpleTOM Contrast** (High ToM vs Low ToM):
- Should identify parameters specifically used for mental state reasoning
- Predicted: Cleaner separation than ToM vs C4
- Predicted: Fewer parameters identified (more selective)
- Predicted: Dampening ToM without much perplexity increase

**Self/Other Contrast**:
- May identify perspective-taking mechanisms
- Less clear if this will strongly affect ToM tasks
- May have broader effects on language use

In [ ]:
# Summarize results for both experiments
def summarize_all_results():
    """Generate summaries for both contrast pair experiments."""
    
    results = {}
    
    for name, output_dir in [("SimpleTOM", EVAL_SIMPLETOM), ("Self/Other", EVAL_SELF_OTHER)]:
        tom_dir = os.path.join(output_dir, "tom")
        summary_csv = os.path.join(output_dir, "tom_summary.csv")
        
        if not os.path.exists(tom_dir):
            print(f"⚠ {name} results not yet generated")
            continue
        
        cmd = [
            "python", "summarize.py",
            "--root", tom_dir,
            "--reps", str(EVAL_REPS),
            "--out_csv", summary_csv
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0:
            print(f"✓ {name} summary saved to {summary_csv}")
            results[name] = summary_csv
        else:
            print(f"ERROR summarizing {name}: {result.stderr}")
    
    return results

# Uncomment to run
# summary_files = summarize_all_results()

In [ ]:
# Visualize and compare results
def plot_contrast_comparison():
    """Plot ToM accuracy and perplexity for both contrast pair methods."""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    experiments = [
        ("SimpleTOM Contrast", EVAL_SIMPLETOM, axes[0]),
        ("Self/Other Contrast", EVAL_SELF_OTHER, axes[1])
    ]
    
    for exp_name, exp_dir, (ax_tom, ax_ppl) in experiments:
        tom_csv = os.path.join(exp_dir, "tom_summary.csv")
        ppl_csv = os.path.join(exp_dir, "perplexity_results.csv")
        
        if not os.path.exists(tom_csv):
            ax_tom.text(0.5, 0.5, f"{exp_name}\nResults not available",
                       ha='center', va='center', transform=ax_tom.transAxes)
            ax_ppl.text(0.5, 0.5, f"{exp_name}\nResults not available",
                       ha='center', va='center', transform=ax_ppl.transAxes)
            continue
        
        # Plot ToM accuracy
        tom_df = pd.read_csv(tom_csv)
        for col in tom_df.columns:
            if col != 'm':
                ax_tom.plot(tom_df['m'], tom_df[col], marker='o', label=col, alpha=0.7)
        
        ax_tom.set_xlabel('m (sparsity level)', fontsize=11)
        ax_tom.set_ylabel('ToM Accuracy', fontsize=11)
        ax_tom.set_title(f'{exp_name} - ToM Performance', fontsize=12, fontweight='bold')
        ax_tom.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        ax_tom.grid(True, alpha=0.3)
        
        # Plot perplexity
        if os.path.exists(ppl_csv):
            ppl_df = pd.read_csv(ppl_csv)
            ax_ppl.plot(ppl_df['m'], ppl_df['perplexity'], marker='s', color='red', linewidth=2)
            ax_ppl.set_xlabel('m (sparsity level)', fontsize=11)
            ax_ppl.set_ylabel('Perplexity', fontsize=11)
            ax_ppl.set_title(f'{exp_name} - WikiText-2 Perplexity', fontsize=12, fontweight='bold')
            ax_ppl.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save figure
    fig_path = os.path.join(PERMANENT_STORAGE, "contrast_comparison.png")
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Comparison plot saved to {fig_path}")
    
    plt.show()

# Uncomment to plot
# plot_contrast_comparison()

## Parameter Overlap Analysis

**Epistemic Status**: Moderate confidence. This analysis assumes gradients have been computed.

Compare which parameters are identified by SimpleTOM vs Self/Other contrasts.

In [ ]:
# Analyze parameter overlap between methods
import torch

def analyze_parameter_overlap(m_value=2e-5):
    """Compare which parameters are masked by different contrast methods."""
    
    # Check if chunks exist
    if not all(os.path.exists(os.path.join(d, "manifest.json")) for d in 
               [SIMPLETOM_HIGH_CHUNKS, SIMPLETOM_LOW_CHUNKS, SELF_CHUNKS, OTHER_CHUNKS]):
        print("⚠ Gradient chunks not yet generated")
        return
    
    # Load manifests
    with open(os.path.join(SIMPLETOM_HIGH_CHUNKS, "manifest.json")) as f:
        manifest = json.load(f)
    
    num_layers = manifest['num_layers']
    module_names = manifest['modules']
    
    # Count masked parameters for each method
    simpletom_masked = 0
    self_other_masked = 0
    overlap_masked = 0
    total_params = 0
    
    for layer_idx in range(num_layers):
        # Load gradients
        simpletom_high = torch.load(os.path.join(SIMPLETOM_HIGH_CHUNKS, f"layer_{layer_idx}.pt"))
        simpletom_low = torch.load(os.path.join(SIMPLETOM_LOW_CHUNKS, f"layer_{layer_idx}.pt"))
        self_grads = torch.load(os.path.join(SELF_CHUNKS, f"layer_{layer_idx}.pt"))
        other_grads = torch.load(os.path.join(OTHER_CHUNKS, f"layer_{layer_idx}.pt"))
        
        for mod_name in module_names:
            # Get gradients
            key = mod_name if mod_name in simpletom_high else mod_name.replace("_proj", "")
            
            g_simpletom_h = simpletom_high[key]
            g_simpletom_l = simpletom_low[key]
            g_self = self_grads[key]
            g_other = other_grads[key]
            
            total_params += g_simpletom_h.numel()
            
            # Compute masks (same logic as evaluation script)
            num_outliers = int(g_simpletom_h.numel() * m_value)
            
            if num_outliers > 0:
                # SimpleTOM mask
                thr_st_h = g_simpletom_h.reshape(-1).topk(k=num_outliers).values[-1]
                thr_st_l = g_simpletom_l.reshape(-1).topk(k=num_outliers).values[-1]
                mask_simpletom = (g_simpletom_h > thr_st_h) & ~(g_simpletom_l > thr_st_l)
                
                # Self/Other mask
                thr_self = g_self.reshape(-1).topk(k=num_outliers).values[-1]
                thr_other = g_other.reshape(-1).topk(k=num_outliers).values[-1]
                mask_self_other = (g_self > thr_self) & ~(g_other > thr_other)
                
                # Count
                simpletom_masked += mask_simpletom.sum().item()
                self_other_masked += mask_self_other.sum().item()
                overlap_masked += (mask_simpletom & mask_self_other).sum().item()
    
    print(f"\nParameter Overlap Analysis (m={m_value})")
    print(f"{'='*60}")
    print(f"Total parameters: {total_params:,}")
    print(f"\nSimpleTOM contrast:")
    print(f"  Masked: {simpletom_masked:,} ({100*simpletom_masked/total_params:.3f}%)")
    print(f"\nSelf/Other contrast:")
    print(f"  Masked: {self_other_masked:,} ({100*self_other_masked/total_params:.3f}%)")
    print(f"\nOverlap:")
    print(f"  Both methods: {overlap_masked:,} ({100*overlap_masked/total_params:.3f}%)")
    
    if simpletom_masked > 0:
        print(f"  % of SimpleTOM params: {100*overlap_masked/simpletom_masked:.1f}%")
    if self_other_masked > 0:
        print(f"  % of Self/Other params: {100*overlap_masked/self_other_masked:.1f}%")
    
    # Jaccard similarity
    union = simpletom_masked + self_other_masked - overlap_masked
    if union > 0:
        jaccard = overlap_masked / union
        print(f"\nJaccard similarity: {jaccard:.3f}")

# Uncomment to run (requires gradient chunks)
# analyze_parameter_overlap(m_value=2e-5)

## Complete Pipeline Runners

In [ ]:
def run_simpletom_pipeline():
    """Run complete SimpleTOM contrast pipeline."""
    print("="*80)
    print("SimpleTOM Contrast Pipeline (High ToM vs Low ToM)")
    print("="*80)
    
    # Prepare datasets
    print("\n[1/5] Preparing datasets...")
    high_path, low_path = prepare_simpletom_datasets()
    
    # Compute gradients
    print("\n[2/5] Computing gradients...")
    compute_contrast_gradients(high_path, SIMPLETOM_HIGH_GRAD, "High ToM")
    compute_contrast_gradients(low_path, SIMPLETOM_LOW_GRAD, "Low ToM")
    
    # Chunk
    print("\n[3/5] Chunking gradients...")
    chunk_gradients(SIMPLETOM_HIGH_GRAD, SIMPLETOM_HIGH_CHUNKS, "High ToM")
    chunk_gradients(SIMPLETOM_LOW_GRAD, SIMPLETOM_LOW_CHUNKS, "Low ToM")
    
    # Evaluate
    print("\n[4/5] Running evaluation...")
    run_evaluation(SIMPLETOM_HIGH_CHUNKS, SIMPLETOM_LOW_CHUNKS, EVAL_SIMPLETOM, "SimpleTOM")
    
    # Summarize
    print("\n[5/5] Summarizing...")
    summarize_all_results()
    
    print("\n" + "="*80)
    print("SimpleTOM Pipeline Complete")
    print("="*80)

def run_self_other_pipeline():
    """Run complete Self/Other contrast pipeline."""
    print("="*80)
    print("Self/Other Contrast Pipeline")
    print("="*80)
    
    # Prepare datasets
    print("\n[1/5] Preparing datasets...")
    self_p, other_p = prepare_self_other_datasets()
    
    # Compute gradients
    print("\n[2/5] Computing gradients...")
    compute_contrast_gradients(self_p, SELF_GRAD, "Self")
    compute_contrast_gradients(other_p, OTHER_GRAD, "Other")
    
    # Chunk
    print("\n[3/5] Chunking gradients...")
    chunk_gradients(SELF_GRAD, SELF_CHUNKS, "Self")
    chunk_gradients(OTHER_GRAD, OTHER_CHUNKS, "Other")
    
    # Evaluate
    print("\n[4/5] Running evaluation...")
    run_evaluation(SELF_CHUNKS, OTHER_CHUNKS, EVAL_SELF_OTHER, "Self/Other")
    
    # Summarize
    print("\n[5/5] Summarizing...")
    summarize_all_results()
    
    print("\n" + "="*80)
    print("Self/Other Pipeline Complete")
    print("="*80)

# Uncomment to run (REQUIRES GPU)
# run_simpletom_pipeline()
# run_self_other_pipeline()

## Final Assessment: Will Dampening Work?

### Epistemic Status: Moderate-to-Low Confidence

**Predicted outcomes for SimpleTOM contrast:**

**Best case scenario** (30% probability):
- Parameters are cleanly separable between high/low ToM
- Masking top 2-5% of parameters causes 20-40% drop in ToM accuracy
- Perplexity increases <10%
- Model maintains general capabilities but struggles with mental state reasoning
- **Interactive dampening works**: Model fails false belief tasks, misattributes knowledge

**Likely scenario** (50% probability):
- Parameters show some separation but substantial overlap
- Masking causes 10-25% drop in ToM accuracy
- Perplexity increases 5-20%
- **Partial dampening**: Model sometimes fails ToM but inconsistently
- Collateral damage to related reasoning (counterfactuals, hypotheticals)

**Worst case scenario** (20% probability):
- ToM is too distributed across parameter space
- No clear separation between conditions
- Either: (1) no ToM effect, or (2) massive perplexity increase
- **Dampening doesn't work** or breaks general language ability

### Why This Might Not Work

1. **Distributed representations**: ToM may not be localized to specific parameters
2. **Training data bleed**: Contrast pairs may not fully separate circuits due to shared training data
3. **Evaluation mismatch**: ToM tasks may use different circuits than training data
4. **Redundancy**: LLMs may have redundant ToM circuits

### Validation Strategy

To test if dampening works:
1. Load masked model and run interactive tests
2. Test false belief understanding in conversation
3. Test if model incorrectly attributes its knowledge to others
4. Compare with control tests (factual reasoning, basic language)

### Recommendations

If dampening is weak:
- Try higher m values (5e-5 to 1e-4)
- Try different masking strategies (multiplicative scaling instead of mean replacement)
- Combine both SimpleTOM AND Self/Other masks
- Use activation steering instead of parameter perturbation